# Phase 3 -- 1024px resolution test (Kaggle-only)

The last untested point of the microaneurysm-resolution hypothesis (docs/07_PHASE3_RESULTS.md,
Result 1 and Result 5): resolution was refuted at 384/512/768px, but 1024px was never tried
because it needs more memory than this project's local machine has. This notebook reproduces
the exact same baseline configuration at 1024px on Kaggle's free GPU quota.

**Before running:** attach the APTOS 2019 competition data via *Add Data -> Competitions ->
"aptos2019-blindness-detection"*, and set the accelerator to a GPU (Settings -> Accelerator).

In [ ]:
!git clone https://github.com/adarshcod30/Diabetic-Retinopathy-Detection.git
%cd Diabetic-Retinopathy-Detection
!pip install -q -e .

In [ ]:
import os
import shutil

# Point this project's expected raw-data layout at Kaggle's attached copy,
# rather than re-downloading ~10 GB that is already sitting in /kaggle/input.
SRC = "/kaggle/input/aptos2019-blindness-detection"
os.makedirs("data/raw/aptos", exist_ok=True)
if not os.path.exists("data/raw/aptos/train_images"):
    os.symlink(f"{SRC}/train_images", "data/raw/aptos/train_images")
if not os.path.exists("data/raw/aptos/train.csv"):
    shutil.copy(f"{SRC}/train.csv", "data/raw/aptos/train.csv")

print("train_images:", len(os.listdir("data/raw/aptos/train_images")), "files")

In [ ]:
# Same preprocessing chain as every other resolution tested (crop -> circle_crop
# -> Ben Graham -> resize) -- only the target size differs.
!python scripts/preprocess.py --dataset aptos --size 1024 --workers 4

In [ ]:
# Identical to the baseline (lr, warmup, patience, CE loss) -- only size and
# batch size change, since 512px's batch 4 was itself set by the LOCAL
# machine's memory ceiling, not by anything about the model or the task.
# Kaggle's T4/P100 has far more headroom (16 GB vs. this project's constrained
# unified-memory budget, docs/05_PROTOTYPE_SCOPE.md Sec.6.2) -- batch 4 below
# is a conservative starting point, not a measured optimum. If the first few
# epochs run well under the GPU's memory limit (check via `nvidia-smi` in a
# cell, or Kaggle's own GPU-memory graph), raise --batch-size (8, then 16) for
# a faster run -- there is no other reason to keep it at 4 here.
!python scripts/train.py --size 1024 --batch-size 4 --run-name kaggle_1024px_ce

## Bringing the result back

Run the cell below, then download `/kaggle/working/kaggle_1024px_results.zip` from the notebook's
Output tab. Unzip it into this repo's `runs/` and `models/checkpoints/` (matching paths), then run
`scripts/evaluate.py` and `scripts/compare.py` locally against the true baseline checkpoint exactly
as every other Phase 3 result was compared -- keep that step local, not on Kaggle, so it stays
consistent with how every other row in the ablation table was produced.

In [ ]:
!zip -r /kaggle/working/kaggle_1024px_results.zip runs/kaggle_1024px_ce models/checkpoints/kaggle_1024px_ce_fold0